# New tree

Having come across a homologous protein in *Escherichia albertii*, I decided I should make a more representative tree that contains more distant homologues.

A BLASTp search in the *nr* database yielded hits with corresponding *E. coli* serotypes, and hits on *E. albertii*, so I will use this to make the tree going forwards.

First, I will use CD-HIT to group related proteins, as the BLAST output contains 5000 hits.

In [30]:
# CD-HIT at 95% identity

!cd-hit -i ../../new_tree_wd/blast/seqdump_nr.fasta -o ../../new_tree_wd/CD-HIT/seqdump.cdhit99.fasta -c 0.99 -n 5
!cd-hit -i ../../new_tree_wd/blast/seqdump_nr.fasta -o ../../new_tree_wd/CD-HIT/seqdump.cdhit95.fasta -c 0.95 -n 5

Program: CD-HIT, V4.8.1 (+OpenMP), Apr 24 2025, 21:58:32
Command: cd-hit -i ../../new_tree_wd/blast/seqdump_nr.fasta -o
         ../../new_tree_wd/CD-HIT/seqdump.cdhit99.fasta -c 0.99
         -n 5

Started: Tue Nov 11 15:47:15 2025
                            Output                              
----------------------------------------------------------------
total seq: 5000
longest and shortest : 543 and 41
Total letters: 1687480
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 2M
Buffer          : 1 X 16M = 16M
Table           : 1 X 65M = 65M
Miscellaneous   : 0M
Total           : 83M

Table limit with the given memory limit:
Max number of representatives: 1380802
Max number of word counting entries: 89500670

comparing sequences from          0  to       5000
.....
     5000  finished        303  clusters

Approximated maximum memory consumption: 84M
writing new database
writing clustering information
program completed !

Total CPU time 0.30
Pr

In [31]:
# Seeing how many clusters are created - trying to make a manageable dataset
!grep -c '^>Cluster' ../../new_tree_wd/CD-HIT/seqdump.cdhit99.fasta.clstr
!grep -c '^>Cluster' ../../new_tree_wd/CD-HIT/seqdump.cdhit95.fasta.clstr

303
54


In [32]:
!mafft --auto ../../new_tree_wd/CD-HIT/seqdump.cdhit99.fasta > ../../new_tree_wd/mafft/seqdump.cdhit99.mafft.fasta
!mafft --auto ../../new_tree_wd/CD-HIT/seqdump.cdhit95.fasta > ../../new_tree_wd/mafft/seqdump.cdhit95.mafft.fasta

nthread = 0
nthreadpair = 0
nthreadtb = 0
ppenalty_ex = 0
stacksize: 8176 kb
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00



Making a distance matrix ..

There are 13 ambiguous characters.
  301 / 303
done.

Constructing a UPGMA tree (efffree=0) ... 
  300 / 303
done.

Progressive alignment 1/2... 
STEP   157 / 302 
Reallocating..done. *alloclen = 2088
STEP   302 / 302  h
done.

Making a distance matrix from msa.. 
  300 / 303
done.

Constructing a UPGMA tree (efffree=1) ... 
  300 / 303
done.

Progressive alignment 2/2... 
STEP   175 / 302 
Reallocating..done. *alloclen = 2095
STEP   302 / 302  h
done.

disttbfast (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)

distout=h
rescale = 1
dndpre (aa) Version 7.526
alg=X, model=BLOSUM62, 1.53, +0.12, -0.00, noshift, amax=0.0
0 thread(s)

minimumweight = 0.000010
autosubalignment = 0.000000
nthread = 0
randomseed = 0
blosum 62 / kimura 200
poffset = 0
niter = 2
sueff_global = 0.100000
nadd = 2
res

Now I will look in Jalview, and use trimAl to remove incomplete/gappy sequences

There are a lot of truncated sequences outwith *E. coli* and *E. albertii* that I aim to remove. However there are a few sequences outwith these groups that are more complete that I want to retain.

Trimal wasn't working properly due to sequences containing ambiguous amino acids (Z and J), so I replaced them with X

In [36]:
!sed -E 's/[ZBJ]/X/g' ../../new_tree_wd/mafft/seqdump.cdhit99.mafft.fasta | sed 's/\*//g' > ../../new_tree_wd/mafft/seqdump.cdhit99.mafft.clean.fasta
!sed -E 's/[ZBJ]/X/g' ../../new_tree_wd/mafft/seqdump.cdhit95.mafft.fasta | sed 's/\*//g' > ../../new_tree_wd/mafft/seqdump.cdhit95.mafft.clean.fasta

In [48]:
!trimal -in ../../new_tree_wd/mafft/seqdump.cdhit95.mafft.clean.fasta -out ../../new_tree_wd/trimal/seqdump.cdhit95.trimal.fasta -resoverlap 0.5 -seqoverlap 50 -htmlout ../../new_tree_wd/trimal/seqdump.cdhit95.trimal.fasta.html

In [49]:
!trimal -in ../../new_tree_wd/trimal/seqdump.cdhit95.trimal.fasta -out ../../new_tree_wd/trimal/seqdump.cdhit95.trimalgo.fasta -gappyout -htmlout ../../new_tree_wd/trimal/seqdump.cdhit95.trimal.fasta.html



For some reason, trimal on the 99% CD-HIT removes a load of columns, leaving each sequence only 40 aa long. From here I will just use the 95% CD-HIT data.
Automated1 does not sufficiently clean the data, so I will have to set the -seqoverlap and -resoverlap manually.

I used -resoverlap 0.5 and -seqoverlap 50, followed by gappyout. I found that -keepheader resulted in some sequence IDs changing, so I removed that.

Now time for RAxML-NG

In [50]:
# Best tree
!raxml-ng --search --msa ../../new_tree_wd/trimal/seqdump.cdhit95.trimalgo.fasta --model BLOSUM62+G4 --prefix ../../new_tree_wd/raxml/new_tree --threads auto


RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 11-Nov-2025 16:39:20 as follows:

raxml-ng --search --msa ../../new_tree_wd/trimal/seqdump.cdhit95.trimalgo.fasta --model BLOSUM62+G4 --prefix ../../new_tree_wd/raxml/new_tree --threads auto

Analysis options:
  run mode: ML tree search
  start tree(s): random (10) + parsimony (10)
  random seed: 1762879160
  tip-inner: OFF
  pattern compression: ON
  per-rate scalers: OFF
  site repeats: ON
  logLH epsilon: general: 10.000000, brlen-triplet: 1000.000000
  fast spr radius: AUTO
  spr subtree cutoff: 1.000000
  fast CLV updates: ON
  branch l

In [55]:
!raxml-ng --bootstrap --msa ../../new_tree_wd/trimal/seqdump.cdhit95.trimalgo.fasta --model BLOSUM62+G4 --bs-trees 100 --prefix ../../new_tree_wd/raxml/new_tree_bs --threads auto


RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 11-Nov-2025 16:56:29 as follows:

raxml-ng --bootstrap --msa ../../new_tree_wd/trimal/seqdump.cdhit95.trimalgo.fasta --model BLOSUM62+G4 --bs-trees 100 --prefix ../../new_tree_wd/raxml/new_tree_bs --threads auto

Analysis options:
  run mode: Bootstrapping
  start tree(s): 
  bootstrap replicates: parsimony (100)
  random seed: 1762880189
  tip-inner: OFF
  pattern compression: ON
  per-rate scalers: OFF
  site repeats: ON
  logLH epsilon: general: 0.100000, brlen-triplet: 1000.000000
  fast spr radius: AUTO
  spr subtree cutoff: 1.000000
  

In [59]:
!raxml-ng --support --tree ../../new_tree_wd/raxml/new_tree.raxml.bestTree --bs-trees ../../new_tree_wd/raxml/new_tree_bs.raxml.bootstraps --prefix ../../new_tree_wd/raxml/new_tree_support


RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 11-Nov-2025 17:08:46 as follows:

raxml-ng --support --tree ../../new_tree_wd/raxml/new_tree.raxml.bestTree --bs-trees ../../new_tree_wd/raxml/new_tree_bs.raxml.bootstraps --prefix ../../new_tree_wd/raxml/new_tree_support

Analysis options:
  run mode: Compute bipartition support (Felsenstein Bootstrap)
  start tree(s): user
  random seed: 1762880926
  SIMD kernels: SSE3
  parallelization: coarse-grained (auto), PTHREADS (auto)

Reading reference tree from file: ../../new_tree_wd/raxml/new_tree.raxml.bestTree
Reference tree size: 30

Reading